In [2]:
import os

# Setting current working directory to where the notebook and run_pipeline.py are
os.chdir("D:/GoogleDrive/exercises/PPRL_applications_for_evaluation/personal_pprl_bloom_filters")

# Verifying
print("Current directory:", os.getcwd())

# Running the pipeline
%run run_pipeline.py

Current directory: D:\GoogleDrive\exercises\PPRL_applications_for_evaluation\personal_pprl_bloom_filters
Starting Bloom Filter Based PPRL Pipeline...

Running: 00_synthetic_data_generation.ipynb


Executing:   0%|          | 0/25 [00:00<?, ?cell/s]

Executing: 100%|██████████| 25/25 [00:04<00:00,  6.17cell/s]



Running: 01_data_preparation.ipynb


Executing: 100%|██████████| 8/8 [00:06<00:00,  1.27cell/s]


...


Running: 04_similarity_and_linkage.ipynb


Executing: 100%|██████████| 16/16 [00:07<00:00,  2.21cell/s]



Running: 05_scalable_pipeline.ipynb


Executing: 100%|██████████| 11/11 [00:03<00:00,  3.09cell/s]


Pipeline completed successfully!
Start time: 2026-03-27 11:51:02.438502
End time  : 2026-03-27 11:51:33.544811
Duration  : 0:00:31.106309
Executed notebooks saved to: D:\GoogleDrive\exercises\PPRL_applications_for_evaluation\personal_pprl_bloom_filters\notebooks_executed


### Create Q-grams

In [ ]:
def get_qgrams(text, q=2):
    text = text.lower().replace(" ", "")
    return [text[i:i+q] for i in range(len(text) - q + 1)]

In [ ]:
get_qgrams("John")

### Creating a bloom filter

In [ ]:
import hashlib
import numpy as np

def hash_qgram(qgram, seed, size):
    return int(hashlib.md5((qgram + str(seed)).encode()).hexdigest(), 16) % size

In [ ]:
def create_bloom_filter(record, size=100, num_hashes=3):
    bf = np.zeros(size, dtype=int)

    # Combine attributes
    combined = record["name"] + record["dob"]
    qgrams = get_qgrams(combined)

    for qg in qgrams:
        for i in range(num_hashes):
            index = hash_qgram(qg, i, size)
            bf[index] = 1

    return bf

### Encode dataset

In [ ]:
bf_A = [create_bloom_filter(r) for r in records_A]
bf_B = [create_bloom_filter(r) for r in records_B]

### Compare bloom filters

In [ ]:
def dice_similarity(bf1, bf2):
    intersection = np.sum(bf1 * bf2)
    return (2 * intersection) / (np.sum(bf1) + np.sum(bf2))

In [ ]:
for i, a in enumerate(bf_A):
    for j, b in enumerate(bf_B):
        sim = dice_similarity(a, b)
        print(f"A{i} vs B{j}: {sim:.3f}")